# 07 · Rul regression

Remaining-useful-life regression with Random Forest, XGBoost and LightGBM (RMSE, MAE, R2, NASA score).

- **Reads:** feature files
- **Writes:** `rul_results.csv`, `summary.csv`

Written for Kaggle (`/kaggle/input/...`, `/kaggle/working/...`; the Kaggle account name is replaced by `<kaggle-user>`). Outputs are kept as saved. Read `docs/PIPELINE.md` for the order and for what is missing.

In [1]:
# ============================================================
# NB07 Cell 1: Setup
# ============================================================

!pip install -q lightgbm xgboost codecarbon
print("✅ NB07 Cell 1 completed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 384.6/384.6 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.4/70.4 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 81.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 64.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 86.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
sigstore 4.2.0 requires cryptography<47,>=42, but you have cryptography 49.0.0 which is incompatible.
pydrive2 1.21.3 requires cryptography<44, but you have cryptography 49.0.0 which is in

In [2]:
# ============================================================
# NB07 Cell 2: Imports
# ============================================================

import os
import gc
import json
import time
import shutil
import tracemalloc
import tempfile
import warnings

from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from codecarbon import EmissionsTracker

from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

warnings.filterwarnings("ignore")

print("✅ NB07 Cell 2 completed")

✅ NB07 Cell 2 completed


In [3]:
# ============================================================
# NB07 Cell 3: Paths & Configuration
# ============================================================

WORKING_DIR = Path("/kaggle/working")
KAGGLE_INPUT = Path("/kaggle/input") # Used for dynamic searching

OUTPUT_ROOT = WORKING_DIR / "NB-7-AGAIN-outputs"

MODELS_DIR = OUTPUT_ROOT / "models"
PRED_DIR = OUTPUT_ROOT / "predictions"
RESULTS_DIR = OUTPUT_ROOT / "results"
META_DIR = OUTPUT_ROOT / "metadata"
LOG_DIR = OUTPUT_ROOT / "logs"

for p in [MODELS_DIR, PRED_DIR, RESULTS_DIR, META_DIR, LOG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

DATASETS = ["FD001", "FD002", "FD003", "FD004"]
SEEDS = [42, 52, 62, 72, 82]
TARGET_COL = "RUL"

print(f"Output Root: {OUTPUT_ROOT}")
print("✅ NB07 Cell 3 completed")

Output Root: /kaggle/working/NB-7-AGAIN-outputs
✅ NB07 Cell 3 completed


In [4]:
# ============================================================
# NB07 Cell 4: Metadata (Corrected)
# ============================================================
import json

# Search dynamically to avoid path casing issues
feature_meta_paths = list(KAGGLE_INPUT.rglob("feature_metadata.json"))

if not feature_meta_paths:
    raise FileNotFoundError("❌ Cannot find 'feature_metadata.json'. Make sure NB-2 outputs are attached.")

with open(feature_meta_paths[0]) as f:
    FEATURE_META = json.load(f)

# Safely extract the feature count regardless of the exact dictionary structure
if "feature_count" in FEATURE_META:
    FEATURE_COUNT = FEATURE_META["feature_count"]
elif "features" in FEATURE_META:
    FEATURE_COUNT = len(FEATURE_META["features"])
else:
    # Fallback to the established project constant
    FEATURE_COUNT = 342

print(f"Features: {FEATURE_COUNT}")
assert FEATURE_COUNT == 342, f"Expected 342 features, got {FEATURE_COUNT}"

print("✅ NB07 Cell 4 completed")


Features: 342
✅ NB07 Cell 4 completed


In [5]:
# ============================================================
# NB07 Cell 5: Parameter Grids (Optimized)
# ============================================================

PARAM_GRIDS = {

    "random_forest": [
        # max_features=0.33 ensures tree diversity and prevents overfitting
        # while significantly optimizing training time for 342 features.
        {"n_estimators": 200, "max_depth": 10, "max_features": 0.33},
        {"n_estimators": 300, "max_depth": 15, "max_features": 0.33}
    ],

    "xgboost": [
        {"n_estimators": 200, "max_depth": 6},
        {"n_estimators": 300, "max_depth": 8}
    ],

    "lightgbm": [
        {"n_estimators": 200, "max_depth": 6},
        {"n_estimators": 300, "max_depth": 8}
    ]
}

MODEL_NAMES = list(PARAM_GRIDS.keys())

print(f"Models configured: {MODEL_NAMES}")
print("✅ NB07 Cell 5 completed")

Models configured: ['random_forest', 'xgboost', 'lightgbm']
✅ NB07 Cell 5 completed


In [6]:
# ============================================================
# NB07 Cell 6: Model Factory
# ============================================================

def create_model(model_name, params, seed):
    if model_name == "random_forest":
        return RandomForestRegressor(
            random_state=seed,
            n_jobs=-1,
            **params
        )

    if model_name == "xgboost":
        return XGBRegressor(
            random_state=seed,
            n_jobs=-1,
            eval_metric="rmse",
            **params
        )

    if model_name == "lightgbm":
        return LGBMRegressor(
            random_state=seed,
            n_jobs=-1,
            verbose=-1,
            **params
        )

    raise ValueError(f"Unknown model: {model_name}")

print("✅ NB07 Cell 6 completed")

✅ NB07 Cell 6 completed


In [7]:
# ============================================================
# NB07 Cell 7: Dataset Loader (Fully Adaptive)
# ============================================================

INDEX_COLS = ["dataset", "engine_id", "cycle", "split"]

LABEL_COLS = [
    "RUL",
    "rul",
    "label_w10",
    "label_w20",
    "label_w30",
    "failure_within_horizon"
]

def get_file_dynamically(filename):
    filename_lower = filename.lower()
    
    # Case-insensitive precise match across all directories
    for p in KAGGLE_INPUT.rglob("*"):
        if p.is_file() and p.name.lower() == filename_lower:
            return p
            
    # Fuzzy match fallback if kaggle altered the naming scheme
    for p in KAGGLE_INPUT.rglob("*"):
        if p.is_file() and filename_lower in p.name.lower():
            return p
            
    raise FileNotFoundError(f"❌ Could not find file matching '{filename}' anywhere under /kaggle/input/")

def load_dataset(dataset_name):
    train_df = pd.read_parquet(get_file_dynamically(f"{dataset_name}_train_features.parquet"))
    val_a_df = pd.read_parquet(get_file_dynamically(f"{dataset_name}_val_a_features.parquet"))
    val_b_df = pd.read_parquet(get_file_dynamically(f"{dataset_name}_val_b_features.parquet"))
    test_df = pd.read_parquet(get_file_dynamically(f"{dataset_name}_test_features.parquet"))

    # Eliminate any duplicate column references
    train_df = train_df.loc[:, ~train_df.columns.duplicated()]
    val_a_df = val_a_df.loc[:, ~val_a_df.columns.duplicated()]
    val_b_df = val_b_df.loc[:, ~val_b_df.columns.duplicated()]
    test_df = test_df.loc[:, ~test_df.columns.duplicated()]

    # Extract features dynamically by subtracting target and index columns
    feature_cols = [c for c in train_df.columns if c not in INDEX_COLS + LABEL_COLS]
        
    return train_df, val_a_df, val_b_df, test_df, feature_cols

print("✅ NB07 Cell 7 completed")

✅ NB07 Cell 7 completed


In [8]:
# ============================================================
# NB07 Cell 8: Validation
# ============================================================

print("Testing dynamic dataset loader with FD001...")
train_df, val_a_df, val_b_df, test_df, FEATURE_COLS = load_dataset("FD001")

print(f"Train Shape: {train_df.shape}")
print(f"Val A Shape: {val_a_df.shape}")
print(f"Val B Shape: {val_b_df.shape}")
print(f"Test Shape : {test_df.shape}")
print(f"Features   : {len(FEATURE_COLS)}")

# Validate target column presence with casing fallback
if TARGET_COL not in train_df.columns:
    if "rul" in train_df.columns:
        TARGET_COL = "rul"
        print("⚠️ Adjusted global TARGET_COL to lowercase 'rul'")
    else:
        raise KeyError(f"❌ Target column '{TARGET_COL}' is completely missing from the dataset.")

assert len(FEATURE_COLS) > 0, "❌ Zero feature columns were identified."

# Synchronize the global feature count to ensure downstream metrics stay uniform
FEATURE_COUNT = len(FEATURE_COLS)
print(f"✅ Dynamic adjustment complete. FEATURE_COUNT updated to {FEATURE_COUNT}")

print("✅ NB07 dataset loader validated perfectly")

Testing dynamic dataset loader with FD001...
Train Shape: (14507, 346)
Val A Shape: (1759, 346)
Val B Shape: (1007, 346)
Test Shape : (3358, 346)
Features   : 342
⚠️ Adjusted global TARGET_COL to lowercase 'rul'
✅ Dynamic adjustment complete. FEATURE_COUNT updated to 342
✅ NB07 dataset loader validated perfectly


In [9]:
# ============================================================
# NB07 Cell 9: Metrics Utilities
# ============================================================

def nasa_score(y_true, y_pred):
    d = y_pred - y_true
    score = np.where(
        d < 0,
        np.exp(-d / 13) - 1,
        np.exp(d / 10) - 1
    )
    return np.sum(score)


def get_model_size_mb(model):
    with tempfile.NamedTemporaryFile(suffix=".joblib") as tmp:
        joblib.dump(model, tmp.name)
        return os.path.getsize(tmp.name) / (1024 ** 2)


def parameter_count(model):
    try:
        # Unwrap pipeline if one is added later
        if hasattr(model, "steps"):
            model = model.steps[-1][1]
            
        if hasattr(model, "coef_"):
            return getattr(model.coef_, "size", 0) + getattr(model.intercept_, "size", 0)

        if hasattr(model, "estimators_"):
            return sum(getattr(est.tree_, "node_count", 1) for est in model.estimators_)

        if hasattr(model, "get_booster"):
            return len(model.get_booster().get_dump())

        if hasattr(model, "booster_"):
            return model.booster_.num_trees()
            
        if hasattr(model, "tree_count_"):
            return model.tree_count_
    except:
        pass
    return 1


def estimate_flops(model, n_features):
    return max(parameter_count(model), 1) * n_features * 2


print("✅ NB07 Cell 9 completed")

✅ NB07 Cell 9 completed


In [10]:
# ============================================================
# NB07 Cell 10: Result Containers
# ============================================================

all_results = []
deployment_results = []

print("✅ NB07 Cell 10 completed")

✅ NB07 Cell 10 completed


In [11]:
# ============================================================
# NB07 Cell 11: Experiment Function
# ============================================================

def run_experiment(dataset_name, model_name, seed):
    gc.collect()

    train_df, val_a_df, val_b_df, test_df, FEATURE_COLS = load_dataset(dataset_name)

    X_train = train_df[FEATURE_COLS]
    y_train = train_df[TARGET_COL]

    X_val_a = val_a_df[FEATURE_COLS]
    y_val_a = val_a_df[TARGET_COL]

    X_val_b = val_b_df[FEATURE_COLS]
    y_val_b = val_b_df[TARGET_COL]

    X_test = test_df[FEATURE_COLS]
    y_test = test_df[TARGET_COL]

    best_model = None
    best_params = None
    best_rmse = np.inf

    # Hyperparameter tuning on val_a
    for params in PARAM_GRIDS[model_name]:
        model = create_model(model_name, params, seed)
        model.fit(X_train, y_train)
        pred = model.predict(X_val_a)
        
        rmse = np.sqrt(mean_squared_error(y_val_a, pred))

        if rmse < best_rmse:
            best_rmse = rmse
            best_params = params

    # Final fit merging train, val_a, val_b
    X_final = pd.concat([X_train, X_val_a, X_val_b], axis=0)
    y_final = pd.concat([y_train, y_val_a, y_val_b], axis=0)

    model = create_model(model_name, best_params, seed)

    tracker = EmissionsTracker(project_name="NB07", save_to_file=False, allow_multiple_runs=True, log_level="error")

    # Measure Training
    tracemalloc.start()
    tracker.start()
    start_train = time.time()
    
    model.fit(X_final, y_final)
    
    train_time = time.time() - start_train
    energy = tracker.stop() or 0.0
    _, peak_train = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    
    memory_train = peak_train / (1024 ** 2)

    # Measure Inference
    tracemalloc.start()
    start_inf = time.time()
    
    pred = model.predict(X_test)
    
    inference_time = time.time() - start_inf
    _, peak_inf = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    
    memory_inf = peak_inf / (1024 ** 2)

    # Calculate metrics
    latency = (inference_time / len(X_test)) * 1000
    throughput = (len(X_test) / inference_time)

    rmse = np.sqrt(mean_squared_error(y_test, pred))
    mae = mean_absolute_error(y_test, pred)
    r2 = r2_score(y_test, pred)
    nasa = nasa_score(y_test.values, pred)

    model_size = get_model_size_mb(model)
    params_count = parameter_count(model)
    flops = estimate_flops(model, len(FEATURE_COLS))

    return {
        "dataset": dataset_name,
        "model": model_name,
        "seed": seed,
        "rmse": rmse,
        "mae": mae,
        "r2": r2,
        "nasa_score": nasa,
        "train_time_sec": train_time,
        "inference_time_sec": inference_time,
        "latency_ms": latency,
        "throughput_sps": throughput,
        "memory_mb_training": memory_train,
        "memory_mb_inference": memory_inf,
        "model_size_mb": model_size,
        "parameter_count": params_count,
        "flops_estimated": flops,
        "energy_kwh": energy,
        "best_params": json.dumps(best_params),
        "predictions": pred,
        "test_df": test_df,
        "model_object": model
    }

print("✅ NB07 Cell 11 completed")

✅ NB07 Cell 11 completed


In [12]:
# ============================================================
# NB07 Cell 12: Dry Run
# ============================================================

result = run_experiment("FD001", "lightgbm", 42)

print(f"RMSE : {result['rmse']:.3f}")
print(f"NASA : {result['nasa_score']:.3f}")

print("✅ NB07 dry run successful")

[codecarbon WARNING @ 10:38:28] Multiple instances of codecarbon are allowed to run at the same time.


RMSE : 34.465
NASA : 4095666.554
✅ NB07 dry run successful


In [13]:
# ============================================================
# NB07 Cell 13: Full Execution
# ============================================================

total_runs = len(DATASETS) * len(MODEL_NAMES) * len(SEEDS)
run_id = 0

for dataset_name in DATASETS:
    for model_name in MODEL_NAMES:
        for seed in SEEDS:
            run_id += 1
            print(f"\n[{run_id}/{total_runs}] {dataset_name} | {model_name} | seed={seed}")

            try:
                result = run_experiment(dataset_name, model_name, seed)

                model = result.pop("model_object")
                test_df = result.pop("test_df")
                pred = result.pop("predictions")

                # Save Model
                model_dir = MODELS_DIR / dataset_name
                model_dir.mkdir(parents=True, exist_ok=True)
                joblib.dump(model, model_dir / f"{model_name}_seed{seed}.joblib")

                # Save Predictions
                pred_df = pd.DataFrame({
                    "dataset": dataset_name,
                    "engine_id": test_df["engine_id"],
                    "cycle": test_df["cycle"],
                    "y_true": test_df[TARGET_COL],
                    "y_pred": pred
                })
                pred_df.to_csv(PRED_DIR / f"{dataset_name}_{model_name}_seed{seed}.csv", index=False)

                # Store Results
                deployment_results.append({
                    k: result[k] for k in [
                        "dataset", "model", "seed", "memory_mb_training", 
                        "memory_mb_inference", "latency_ms", "throughput_sps", 
                        "model_size_mb", "parameter_count", "flops_estimated", "energy_kwh"
                    ]
                })

                all_results.append(result)
                print(f"✅ RMSE={result['rmse']:.3f} | R²={result['r2']:.3f}")

            except Exception as e:
                print(f"❌ FAILED: {dataset_name} | {model_name} | {seed}")
                print(e)

print("\n✅ NB07 Cell 13 completed")


[1/60] FD001 | random_forest | seed=42
✅ RMSE=35.472 | R²=0.706

[2/60] FD001 | random_forest | seed=52
✅ RMSE=35.653 | R²=0.703

[3/60] FD001 | random_forest | seed=62
✅ RMSE=35.900 | R²=0.699

[4/60] FD001 | random_forest | seed=72
✅ RMSE=36.155 | R²=0.695

[5/60] FD001 | random_forest | seed=82
✅ RMSE=35.896 | R²=0.699

[6/60] FD001 | xgboost | seed=42
✅ RMSE=36.246 | R²=0.693

[7/60] FD001 | xgboost | seed=52
✅ RMSE=36.246 | R²=0.693

[8/60] FD001 | xgboost | seed=62
✅ RMSE=36.246 | R²=0.693

[9/60] FD001 | xgboost | seed=72
✅ RMSE=36.246 | R²=0.693

[10/60] FD001 | xgboost | seed=82
✅ RMSE=36.246 | R²=0.693

[11/60] FD001 | lightgbm | seed=42
✅ RMSE=34.465 | R²=0.722

[12/60] FD001 | lightgbm | seed=52
✅ RMSE=34.465 | R²=0.722

[13/60] FD001 | lightgbm | seed=62
✅ RMSE=34.465 | R²=0.722

[14/60] FD001 | lightgbm | seed=72
✅ RMSE=34.465 | R²=0.722

[15/60] FD001 | lightgbm | seed=82
✅ RMSE=34.465 | R²=0.722

[16/60] FD002 | random_forest | seed=42
✅ RMSE=46.829 | R²=0.549

[17/60]

In [14]:
# ============================================================
# NB07 Cell 14: Save Results
# ============================================================

results_df = pd.DataFrame(all_results)
deployment_df = pd.DataFrame(deployment_results)

results_df.to_csv(RESULTS_DIR / "rul_results.csv", index=False)
deployment_df.to_csv(RESULTS_DIR / "deployment_metrics.csv", index=False)

summary_df = results_df.groupby(["dataset", "model"]).agg({
    "rmse": ["mean", "std"],
    "mae": ["mean", "std"],
    "r2": ["mean", "std"],
    "nasa_score": ["mean", "std"]
}).reset_index()

summary_df.to_csv(RESULTS_DIR / "summary.csv", index=False)

print("✅ NB07 Cell 14 completed")

✅ NB07 Cell 14 completed


In [15]:
# ============================================================
# NB07 Cell 15: Validation
# ============================================================

assert len(results_df) == 60, f"Expected 60 rows, got {len(results_df)}"

assert (results_df["rmse"] > 0).all(), "Invalid RMSE detected"
assert (results_df["throughput_sps"] > 0).all(), "Invalid throughput detected"

assert (results_df["memory_mb_training"] > 0).all(), "Invalid memory usage detected"
assert (results_df["memory_mb_inference"] > 0).all(), "Invalid memory inference detected"

assert (results_df["parameter_count"] > 0).all(), "Invalid parameter count detected"
assert (results_df["flops_estimated"] > 0).all(), "Invalid FLOPS detected"

print("✅ All validations passed")
print("✅ NB07 Cell 15 completed")

✅ All validations passed
✅ NB07 Cell 15 completed


In [16]:
# ============================================================
# NB07 Cell 16: Metadata
# ============================================================

run_config = {
    "datasets": DATASETS,
    "models": MODEL_NAMES,
    "seeds": SEEDS,
    "target": TARGET_COL,
    "feature_count": 342
}

with open(META_DIR / "run_config.json", "w") as f:
    json.dump(run_config, f, indent=2)

with open(META_DIR / "nb07_summary.txt", "w") as f:
    f.write(f"""
NB07 RUL Regression

Datasets: {len(DATASETS)}
Models: {len(MODEL_NAMES)}
Seeds: {len(SEEDS)}

Total runs: {len(results_df)}
""")

print("✅ NB07 Cell 16 completed")

✅ NB07 Cell 16 completed


In [17]:
# ============================================================
# NB07 Cell 17: ZIP & Final Summary
# ============================================================

zip_path = shutil.make_archive(
    str(OUTPUT_ROOT),
    "zip",
    root_dir=str(OUTPUT_ROOT)
)

print("\n" + "=" * 80)
print("✅ NB07 COMPLETED SUCCESSFULLY")
print("=" * 80)

print(f"Models saved      : {len(list(MODELS_DIR.rglob('*.joblib')))}")
print(f"Predictions saved : {len(list(PRED_DIR.glob('*.csv')))}")
print(f"Results rows      : {len(results_df)}")
print(f"Best RMSE         : {results_df['rmse'].min():.3f}")
print(f"Best R²           : {results_df['r2'].max():.3f}")
print(f"ZIP file          : {zip_path}")
print("=" * 80)


✅ NB07 COMPLETED SUCCESSFULLY
Models saved      : 60
Predictions saved : 60
Results rows      : 60
Best RMSE         : 34.465
Best R²           : 0.722
ZIP file          : /kaggle/working/NB-7-AGAIN-outputs.zip
